# 03. Robustness

**What this notebook establishes.** Everything run to try to break the central claim,
and what survived.

The claim under attack: *most immediate regions cannot be resolved.* If that is an
artefact of a modelling choice rather than a property of the data, one of the checks
below should show it.

In [1]:
import json
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, MODEL, RES = ROOT / "data/processed", ROOT / "2-model", ROOT / "3-results"

def check(label, computed, published, tol=1e-9):
    "Recompute a published value and fail loudly if the manuscript no longer matches."
    ok = abs(float(computed) - float(published)) <= tol
    print(f"{'OK  ' if ok else 'MISMATCH'}  {label}: manuscript={published}  recomputed={computed}")
    assert ok, f"{label}: manuscript says {published}, artefacts say {computed}"

def classified(draws, lo_q=0.025, hi_q=0.975):
    "The single criterion used everywhere in this study: 95% ETI excluding zero."
    lo, hi = np.quantile(draws, lo_q, axis=1), np.quantile(draws, hi_q, axis=1)
    return (lo > 0) | (hi < 0)

def slopes(idata, var="b"):
    return idata.posterior[var].stack(sample=("chain", "draw")).values

## Is the indeterminacy manufactured by the prior?

No. Removing pooling entirely *lowers* the classification rate; a spatial prior raises it
by about one percentage point. Shrinkage is costing classifications here, not creating
them, so the pessimistic reading is not self-inflicted.

Note the honest counterweight in the same table: the minimum detectable slope is
strongly prior-dependent (0.0344 pooled against 0.0632 unpooled, that is 1.54 against
2.84 times the national drift). The manuscript quotes both.

In [2]:
rob = pd.read_csv(RES / "tables/spatial_robustness.csv")
display(rob)
for prior, n in [("nopool", 77), ("exch", 84), ("bym2", 89)]:
    check(f"{prior}", rob[rob.prior == prior].classified.iloc[0], n)

,prior,classified,share,median_sd,median_mde,divergences
0,nopool,77,0.150980,0.032246,0.063201,0
1,exch,84,0.164706,0.017551,0.034400,0
2,bym2,89,0.174510,0.017424,0.034151,0


OK    nopool: manuscript=77  recomputed=77
OK    exch: manuscript=84  recomputed=84
OK    bym2: manuscript=89  recomputed=89


## Does the result depend on summarising each region by a straight line?

A fair question, because a region's own series rejects the log-linear fit in 13.3% of
cases overall and 26.5% in the highest-exposure quintile — which is where the study
actually makes determinations. It does not: a per-region quadratic and a national
second-order random walk move the count by less than the seeds do.

In [3]:
shape = pd.read_csv(RES / "tables/trend_shape.csv")
display(shape)
for tr, n in [("linear", 89), ("quadratic", 83), ("rw", 84)]:
    check(f"trend={tr}", shape[shape.trend == tr].classified.iloc[0], n)

,trend,classified,pct,median_sd,alpha,divergences,worst_rhat
0,linear,89,17.5,0.0175,56.4,0,1.0026
1,quadratic,83,16.3,0.0174,59.1,0,1.0028
2,rw,84,16.5,0.0174,56.4,0,1.0025


OK    trend=linear: manuscript=89  recomputed=89
OK    trend=quadratic: manuscript=83  recomputed=83
OK    trend=rw: manuscript=84  recomputed=84


## How stable is the count itself?

Eight refits of the identical specification. The spread is small and, more importantly,
it lives entirely at the decision boundary: 78 regions classify under every seed, 417
under none, and only 15 ever change status. This is a property of thresholding a
continuous quantity, not of the estimate, which is why the probability of direction is
reported alongside the count.

In [4]:
seeds = json.load(open(RES / "SEED_STABILITY.json"))
print(f"counts across 8 seeds: {seeds['counts']}")
print(f"mean {seeds['mean']}, sd {seeds['sd']}, range {seeds['min']}-{seeds['max']} "
      f"({seeds['range_pct_of_510']}% of 510)")
check("always classified", seeds["regions_classified_in_every_run"], 78)
check("never classified", seeds["regions_classified_in_no_run"], 417)
check("changing status", seeds["regions_that_flip_between_runs"], 15)

counts across 8 seeds: [86, 87, 88, 86, 85, 84, 85, 83]
mean 85.5, sd 1.6, range 83-88 (0.98% of 510)
OK    always classified: manuscript=78  recomputed=78
OK    never classified: manuscript=417  recomputed=417
OK    changing status: manuscript=15  recomputed=15


## Is the variance function doing the work?

Partly, and this one changed a conclusion. A single dispersion parameter imposes the same
relative variability on every unit whatever its size. The posterior predictive check
shows the consequence: adequate fit overall, but a gradient across exposure, with the
model predicting too much variability in the largest units.

Letting dispersion depend on exposure flattens the gradient and barely moves the
classification count — but it widens the spread of precision across units from 1.86-fold
to 3.4-fold. An earlier draft of this paper claimed precision was nearly independent of
unit size. That claim was an artefact of the simpler variance function and has been
removed.

Note the residual-degrees-of-freedom divisor below. Dividing by the number of
observations instead, as a first version of this check did, makes a correctly specified
model look overdispersed.

In [5]:
panel = pd.read_csv(PROC / "panel_region_year.csv").sort_values(["rgi_id", "year"]).reset_index(drop=True)
y = panel.avoidable.values
resid_df = len(panel) - 2 * panel.rgi_id.nunique()

for tag in ("global", "exposure"):
    idata = az.from_netcdf(MODEL / f"idata_disp_{tag}.nc")
    rep = idata["posterior_predictive"]["y"].stack(sample=("chain", "draw")).values
    pearson2 = (y - rep.mean(axis=1)) ** 2 / np.maximum(rep.var(axis=1), 1e-9)
    q = pd.qcut(panel.groupby("rgi_id").avoidable.transform("sum"), 5, labels=False) + 1
    byq = pd.Series(pearson2).groupby(q).mean() * (len(panel) / resid_df)
    print(f"{tag:9} overall {pearson2.sum() / resid_df:.3f}   by quintile {byq.round(2).tolist()}")
    overall = float(pearson2.sum() / resid_df)   # recomputed, not restated
    if tag == "global":
        check("global dispersion, overall chi2/df", round(overall, 3), 0.945)
        check("PPC gradient, lowest quintile", round(float(byq.iloc[0]), 2), 1.04)
        check("PPC gradient, highest quintile", round(float(byq.iloc[-1]), 2), 0.78)
    else:
        check("exposure dispersion, overall chi2/df", round(overall, 3), 0.992)

global    overall 0.945   by quintile [1.04, 1.06, 0.95, 0.91, 0.78]
OK    global dispersion, overall chi2/df: manuscript=0.945  recomputed=0.945
OK    PPC gradient, lowest quintile: manuscript=1.04  recomputed=1.04
OK    PPC gradient, highest quintile: manuscript=0.78  recomputed=0.78


exposure  overall 0.992   by quintile [1.03, 1.06, 0.96, 0.96, 0.95]
OK    exposure dispersion, overall chi2/df: manuscript=0.992  recomputed=0.992


## Can covariates rescue trend detectability?

The small area estimation literature exists to borrow strength, so this has to be shown
rather than asserted. The distinction the design turns on:

- a covariate can genuinely sharpen a **level**;
- a covariate that predicts a **trend** can raise the classification count without any
  change in the underlying data, because part of the resulting claim came from the
  covariate rather than from the deaths.

In this study the slope coefficient turned out to include zero, so there is nothing for
the covariate to assert and the residual reduces to the departure-from-national contrast
already reported. Its count is that same quantity, not an independent result.

Note also the static index has, by construction, **zero within-region variance**, so it
cannot carry information about the direction of change within a region.

In [6]:
path = RES / "tables/covariate_models.csv"
if path.exists():
    cov = pd.read_csv(path)
    display(cov)
    for variant, published in [("base", 89), ("level", 85), ("slope", 89), ("timevar", 103)]:
        check(f"covariate: {variant}",
              int(cov[cov.variant == variant].classified_slope.iloc[0]), published)
    print("\nWithin-region share of covariate variance (from covariates_report.txt):")
    print("\n".join(l for l in (PROC / "covariates_report.txt").read_text().splitlines()
                     if "within-region" in l))
else:
    print("covariate models not yet fitted; run 2-model/14_covariates.py")

,variant,classified_slope,pct_slope,median_sd_slope,median_sd_level,divergences,gamma_a,classified_residual,pct_residual,gamma_b,delta_nicu,delta_ubs,between_nicu,between_ubs,within_nicu,within_ubs
0,base,89,17.5,0.0175,0.0929,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,level,85,16.7,0.0175,0.0914,0,-0.1552 (-0.1913 to -0.1182),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,slope,89,17.5,0.0174,0.0931,0,NaN,11.0,2.2,-0.0023 (-0.0068 to 0.0018),NaN,NaN,NaN,NaN,NaN,NaN
3,timevar,103,20.2,0.0174,0.0929,0,NaN,NaN,NaN,NaN,-0.0174 (-0.0367 to 0.0021),0.0400 (0.0146 to 0.0648),NaN,NaN,NaN,NaN
4,mundlak,103,20.2,0.0174,0.0935,0,NaN,NaN,NaN,NaN,NaN,NaN,-0.0253 (-0.0523 to 0.0001),0.0396 (0.0097 to 0.0696),-0.0072 (-0.0356 to 0.0215),0.0322 (-0.0157 to 0.0796)


OK    covariate: base: manuscript=89  recomputed=89
OK    covariate: level: manuscript=85  recomputed=85
OK    covariate: slope: manuscript=89  recomputed=89
OK    covariate: timevar: manuscript=103  recomputed=103

Within-region share of covariate variance (from covariates_report.txt):
          idhm: mean    0.680  sd   0.069  range   0.491 to    0.824  within-region share of variance 0.000
 nicu_per_1000: mean    1.743  sd   2.520  range   0.000 to   27.726  within-region share of variance 0.119
  ubs_per_1000: mean   27.222  sd  12.105  range   0.300 to  103.234  within-region share of variance 0.088
Note: IDHM is fixed at 2010 and has, by construction, zero within-region


## Multiplicity, and attribution at state level

Two results that were computed and, in an earlier draft, not reported. Both are now in
the manuscript.

The classification criterion makes 510 simultaneous decisions. Hierarchical shrinkage
damps multiplicity, which is why the primary criterion is uncorrected, but the size of
the drop under explicit control bounds how much any single classification can carry.

Attribution at state level depended entirely on the variance function. Under one shared
dispersion no state separated from the national drift; that specification imposes a
common floor on relative variability whatever the unit's size, and the dependence of
dispersion on exposure is strong. Under a fitted exposure-dependent dispersion six states
separate, and unpooled ten do, nesting the same six.

In [7]:
prec = (RES / "03_precision_report.log").read_text()
print("\n".join(l for l in prec.splitlines() if "BH-FDR" in l or "Bonferroni" in l))

bh = {l.split(":")[0].split("]")[0].strip("["): int(l.split(":")[1])
      for l in prec.splitlines() if "surviving BH-FDR" in l}
check("regions surviving BH-FDR, rate", bh["rate"], 10)
check("regions surviving BH-FDR, share", bh["share"], 16)

sd = pd.read_csv(RES / "tables/state_dispersion.csv")
check("states departing, single dispersion", sd.departs_global.sum(), 0)
check("states departing, exposure-dependent dispersion", sd.departs_exposure.sum(), 6)
check("states departing, unpooled", sd.departs_unpooled.sum(), 10)
check("the six are nested in the ten", int((sd.departs_exposure & ~sd.departs_unpooled).sum()), 0)
print(f"\nRio de Janeiro: own {sd[sd.UF=='RJ'].slope_own.iloc[0]:.4f}, "
      f"single-dispersion model {sd[sd.UF=='RJ'].b_global.iloc[0]:.4f}, "
      f"t={sd[sd.UF=='RJ'].t.iloc[0]:.1f}")

[rate] surviving BH-FDR 5% : 10
[rate] Bonferroni 5%       : 5
[share] surviving BH-FDR 5% : 16
[share] Bonferroni 5%       : 10
OK    regions surviving BH-FDR, rate: manuscript=10  recomputed=10
OK    regions surviving BH-FDR, share: manuscript=16  recomputed=16
OK    states departing, single dispersion: manuscript=0  recomputed=0
OK    states departing, exposure-dependent dispersion: manuscript=6  recomputed=6
OK    states departing, unpooled: manuscript=10  recomputed=10
OK    the six are nested in the ten: manuscript=0  recomputed=0

Rio de Janeiro: own -0.0014, single-dispersion model -0.0155, t=5.2


## Alternative explanations that were tested and rejected

Each of these would, if true, undercut the central claim. None does. The p-value on the
coding-quality correlation is nominally significant and is reported rather than omitted;
the effect is far too small to account for the findings, and it points in the direction
of *faster* apparent improvement where coding deteriorated.

In [8]:
s = json.load(open(RES / "SUPPORTING_CHECKS.json"))
print(json.dumps(s["coding_quality"], indent=2))
print(json.dumps(s["outcome_definition"], indent=2))
print(json.dumps(s["common_national_shock"], indent=2))
print(json.dumps(s["winners_curse"], indent=2))

check("avoidable, unpooled (%)", s["outcome_definition"]["avoidable"]["pct"], 16.3)
check("all-cause neonatal, unpooled (%)", s["outcome_definition"]["all_neonatal"]["pct"], 10.4)
check("common national shock share", s["common_national_shock"]["common_variance_share"], 0.0067)
check("winner's curse ratio", s["winners_curse"]["inflation_ratio"], 1.91)

{
  "illdef_share_first": 0.0569,
  "illdef_share_last": 0.0541,
  "illdef_share_min": 0.0511,
  "illdef_share_max": 0.0596,
  "n_regions_tested": 510,
  "spearman_illdef_trend_vs_slope": -0.093,
  "spearman_p": 0.0367
}
{
  "avoidable": {
    "units_fitted": 510,
    "classified": 83,
    "pct": 16.3,
    "median_deaths_per_unit": 115
  },
  "four_group": {
    "units_fitted": 510,
    "classified": 52,
    "pct": 10.2,
    "median_deaths_per_unit": 230
  },
  "all_neonatal": {
    "units_fitted": 510,
    "classified": 53,
    "pct": 10.4,
    "median_deaths_per_unit": 245
  }
}
{
  "regions_used": 142,
  "min_deaths": 200,
  "common_variance_share": 0.0067
}
{
  "median_abs_slope_resolved": 0.039,
  "median_abs_slope_unresolved": 0.0204,
  "inflation_ratio": 1.91
}
OK    avoidable, unpooled (%): manuscript=16.3  recomputed=16.3
OK    all-cause neonatal, unpooled (%): manuscript=10.4  recomputed=10.4
OK    common national shock share: manuscript=0.0067  recomputed=0.0067
OK    winner

## The two quantities the Discussion adds

These are the transferability rule and the completeness bound. Neither is a model fit;
both are arithmetic on quantities already established above, and both are recomputed here
because they are published numbers.

The completeness bound is the one that matters defensively. If capture grows at rate *r*
per year, the fitted slope is displaced by exactly log(1 + *r*), additively in the mean.
That moves where a slope sits without touching how precisely it is estimated, so the
detectability result survives any amount of capture drift while the direction of the
national decline does not. Note the erasing rate is exp(|mu_b|) - 1, not |mu_b|.

In [9]:
t = json.load(open(RES / "TRANSFERABILITY.json"))
print(json.dumps(t, indent=2))

thr = t["threshold_rule"]
check("threshold rule, lowest exposure (deaths/unit-year)", thr["lowest"]["deaths_per_unit_year"], 3.5)
check("threshold rule, lowest exposure power", 100 * thr["lowest"]["power"], 6.1, tol=0.05)
check("threshold rule, median exposure (deaths/unit-year)", thr["median"]["deaths_per_unit_year"], 10.7)
check("threshold rule, median exposure power", 100 * thr["median"]["power"], 9.5, tol=0.05)
check("threshold rule, highest exposure (deaths/unit-year)", thr["highest"]["deaths_per_unit_year"], 37.1)
check("threshold rule, highest exposure power", 100 * thr["highest"]["power"], 23.5, tol=0.05)

cb = t["completeness_bound"]
check("capture drift displacement at 1%/yr", cb["displacement_if_capture_improves_1pct_per_year"], 0.010, tol=5e-5)
check("displacement as share of national drift (%)", 100 * cb["displacement_as_share_of_national_drift"], 45, tol=0.5)
check("capture growth erasing the drift (%/yr)", cb["capture_growth_that_would_erase_the_drift_pct_per_year"], 2.25, tol=0.005)

# The power column must be the same one the design analysis committed, not a re-derivation.
d = pd.read_csv(RES / "tables/design_two_arm.csv")
a1 = d[d.arm.str.startswith("A") & (d.true_slope_x_national == 1.0)].sort_values("exposure_quintile")
assert list(a1.power) == [q["power_vs_national_drift"] for q in thr["quintiles"]],     "threshold rule drifted from the committed design table"
print()
print("OK    threshold rule power column matches design_two_arm.csv")

{
  "threshold_rule": {
    "quintiles": [
      {
        "exposure_quintile": 1,
        "median_avoidable_deaths_per_unit_year": 3.5,
        "power_vs_national_drift": 0.061,
        "type_S": 0.1181,
        "type_M": 5.48
      },
      {
        "exposure_quintile": 2,
        "median_avoidable_deaths_per_unit_year": 7.1,
        "power_vs_national_drift": 0.074,
        "type_S": 0.0933,
        "type_M": 3.79
      },
      {
        "exposure_quintile": 3,
        "median_avoidable_deaths_per_unit_year": 10.7,
        "power_vs_national_drift": 0.095,
        "type_S": 0.0455,
        "type_M": 3.13
      },
      {
        "exposure_quintile": 4,
        "median_avoidable_deaths_per_unit_year": 17.7,
        "power_vs_national_drift": 0.114,
        "type_S": 0.0223,
        "type_M": 2.66
      },
      {
        "exposure_quintile": 5,
        "median_avoidable_deaths_per_unit_year": 37.1,
        "power_vs_national_drift": 0.235,
        "type_S": 0.0083,
        "type_M"

## The planning geography

The ladder uses IBGE's immediate region, an economic geography. The health system plans in
*regiões de saúde*. If the resolving power were an artefact of an administratively
irrelevant partition, the planning partition would relieve it. It does not: more deaths per
unit, marginally narrower intervals, the same conclusion.

In [10]:
h = json.load(open(RES / "HEALTH_REGION.json"))
print(json.dumps(h, indent=2))

check("health regions with data", h["units"], 433)
check("median avoidable deaths per health region", h["median_deaths_per_unit"], 152)
check("health regions classified", h["classified"], 75)
check("health regions classified (%)", h["pct_classified"], 17.3, tol=0.05)
check("health regions classified, unpooled", h["classified_unpooled"], 78)
check("health regions rising, hierarchical", h["rising_hierarchical"], 0)
check("unmapped municipality codes", h["crosswalk_provenance"]["municipality_codes_unmapped"], 23)
check("unmapped births", h["crosswalk_provenance"]["births_unmapped"], 980)
check("unmapped avoidable deaths", h["crosswalk_provenance"]["avoidable_deaths_unmapped"], 0)

# The comparison must be read from the committed ladder, not retyped.
lad = pd.read_csv(RES / "tables/aggregation_ladder.csv").set_index("level")
assert h["comparison_immediate_region"]["classified"] == int(lad.loc["immediate region", "classified"])
print()
print("OK    immediate-region comparison matches aggregation_ladder.csv")

# The crosswalk must fail on exactly the codes the immediate-region crosswalk fails on.
print(f"exposure gain: {100 * (h['median_deaths_per_unit'] / h['comparison_immediate_region']['median_deaths_per_unit'] - 1):.0f}%")

{
  "level": "health region",
  "units": 433,
  "median_deaths_per_unit": 152,
  "classified": 75,
  "pct_classified": 17.3,
  "rising_hierarchical": 0,
  "median_posterior_sd": 0.0169,
  "median_mde": 0.0331,
  "divergences": 0,
  "worst_rhat": 1.0049,
  "units_fitted": 433,
  "classified_unpooled": 78,
  "pct_unpooled": 18.0,
  "rising_unpooled": 7,
  "crosswalk_provenance": {
    "municipality_codes_unmapped": 23,
    "municipality_years_unmapped": 166,
    "births_unmapped": 980,
    "avoidable_deaths_unmapped": 0
  },
  "comparison_immediate_region": {
    "units": 510,
    "median_deaths_per_unit": 115,
    "classified": 81,
    "pct_classified": 15.9,
    "median_posterior_sd": 0.0176,
    "classified_unpooled": 83,
    "pct_unpooled": 16.3
  }
}
OK    health regions with data: manuscript=433  recomputed=433
OK    median avoidable deaths per health region: manuscript=152  recomputed=152
OK    health regions classified: manuscript=75  recomputed=75
OK    health regions classified

## What remains open

Stated here so it is not mistaken for something that was checked.

- **Non-linearity is real and unresolved as a descriptive matter.** The flexible-trend
  fits show the *count* is robust, but 26.5% of the highest-exposure regions still reject
  a straight line. The paper reports the linear summary and says so.
- **Birth under-registration** in parts of the North would inflate denominators and mimic
  improvement. Not addressed here; carried as a limitation.
- **The cause grouping was inherited** from the source panel and is not re-derived in
  this repository.